<a href="https://colab.research.google.com/github/varshamqa/AIML_Practice_Repo_VSC1/blob/main/AIML_LangGraph_VM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain langchain-groq langgraph

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.8/247.8 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.5/160.5 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 681.0/681.0 kB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.0/134.0 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.6/212.6 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.2/320.2 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from google.colab import userdata

In [3]:
from google.colab import userdata

# Correct way to retrieve the secret by its assigned name
try:
    GROC_API_KEY = userdata.get('GROC_API_KEY')
    print("GROC_API_KEY retrieved successfully!")
except userdata.SecretNotFoundError:
    print("Secret GROC_API_KEY is still not found. Please ensure it's added correctly in the Colab secrets panel.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

GROC_API_KEY retrieved successfully!


In [4]:
llm = ChatGroq(
    model = "llama-3.3-70b-versatile",
    temperature= 0.3,
    api_key=GROC_API_KEY # Pass the retrieved API key
)

In [5]:
class StudentState(TypedDict):
  question: str
  category: str
  answer: str
  final_response: str

In [6]:
def analyze_question(state: StudentState):

  question = state["question"]

  prompt = f"""
  Classify the following student questions into exactly ones of these categories:
  1) AI_ML
  2) OTHER

  Question: {question}

  Return the Category Name only
  """

  response = llm.invoke(prompt)

  category = response.content.strip().upper()

  if "AI_ML" in category:
    category = "AI_ML"
  else:
    category = "OTHER"

  return {
      "category": category
  }


In [7]:
def generate_answer(state: StudentState):

  question = state["question"]

  prompt = f"""
  You are an expert AI/ML Professor who is well versed with all modern day and traditional AI/ML concepts
        Explain the technical concepts to the students in a simple way and in a structured manner.
        Follow this format to answer the asked question:

        Answer this question in a beginner manner
        {question}

        1) Definition
        2) Pre-Requisites
        3) Architecture
        4) Core features
        5) Advantages & Disadvantages
        6) Use Cases
        7) Real World Example
        8) One line conclusion

        The explanation given should be considered by the fact that the audience is a beginner.
        """
  response = llm.invoke(prompt)

  return {
      "answer": response.content
  }



In [8]:
def review_answer(state: StudentState):

  answer = state["answer"]

  prompt = f"""
  Review the following answer. Be a strict reviewer
  Check for the below points and judge the answer

  1) Is the answer technically correct
  2) The response should be Beginner - Friendly
  3) Check for the clarification of the answer, it should be clear enough for beginners
  4) Check for un-necessary complexity

  If required, improvise the answer and then return the improved answer.

  Original Answer: {answer}
  """

  response = llm.invoke(prompt)

  return {
      "final_response": response.content
  }

In [9]:
def handle_other(state:StudentState):
  return{
      "final_response": "I am currently designed to answer AI and ML Questions only"
  }

In [10]:
def route_question(state:StudentState):
  if state["category"] == "AI_ML":
    return "generate_answer"
  else:
    return "handle_other"

In [11]:
workflow = StateGraph(StudentState)

In [12]:
workflow.add_node(
    "analyze_question",
    analyze_question
)

workflow.add_node(
    "generate_answer",
    generate_answer
)

workflow.add_node(
    "review_answer",
    review_answer
)

workflow.add_node(
    "handle_other",
    handle_other
)

In [13]:
workflow.add_edge(
    START,
    "analyze_question"
)

In [14]:
workflow.add_conditional_edges(
    "analyze_question",
    route_question,
    {
        "generate_answer": "generate_answer",
        "handle_other": "handle_other"
    }
)

In [15]:
workflow.add_edge(
    "generate_answer",
    "review_answer"
)

In [16]:
workflow.add_edge(
    "review_answer",
    END
)

workflow.add_edge(
    "handle_other",
    END
)

In [17]:
app = workflow.compile()

In [20]:
question = input("Ask your question: ")

result = app.invoke({
    "question": question,
    "category": "",
    "answer": "",
    "final_response": ""
})

Ask your question: Explain Generative AI 


In [21]:
print(result["final_response"])

**Review**

The original answer is well-structured and provides a comprehensive overview of Generative AI. However, there are some areas that can be improved to make it more beginner-friendly and clear.

**Technical Correctness: 8/10**
The answer is technically correct, but some points can be simplified or clarified for beginners. For example, the explanation of the architecture of Generative AI models can be simplified, and the concept of latent space can be introduced more intuitively.

**Beginner-Friendliness: 7/10**
The answer assumes some basic knowledge of machine learning and deep learning concepts, which may not be familiar to all beginners. Additionally, some technical terms, such as convolutional neural networks and optimization algorithms, are not explained in detail.

**Clarity: 8/10**
The answer is generally clear, but some points can be clarified or simplified for beginners. For example, the explanation of the difference between supervised and unsupervised learning can be